# Connect to Neo4j — deluxe-analyze

**Current Neo4j IP:** `34.28.39.13`  
**Bolt port:** `7687`  
**HTTP port:** `7474`

## Firewall / Access requirements

The Neo4j VM firewall (`allow-neo4j-bolt`) whitelists the range `181.78.0.0/24`.  
Your client IP must fall within that range, **or** you must connect through a Cloud IAP tunnel:

```bash
# Open IAP tunnel — run in a terminal, keep it running
gcloud compute start-iap-tunnel neo4j-vm 7687 \n  --local-host-port=localhost:7687 \n  --project=engaged-stage-463123-e0 \n  --zone=us-central1-a
```

Then set `NEO4J_IP = "localhost"` in the cell below.

## Password

Retrieve the password from GCP Secret Manager — do **not** hardcode it:

```bash
gcloud secrets versions access latest \n  --secret=neo4j-password \n  --project=engaged-stage-463123-e0
```

Or set `NEO4J_PASSWORD` as an environment variable before launching the notebook.


In [ ]:
import os
from neo4j import GraphDatabase

# Direct connection — works if your IP is in 181.78.0.0/24
# Use 'localhost' if you have an active Cloud IAP tunnel on port 7687
NEO4J_IP = os.environ.get("NEO4J_IP", "34.28.39.13")

# Set NEO4J_PASSWORD env var before running, or retrieve from Secret Manager:
# gcloud secrets versions access latest --secret=neo4j-password --project=engaged-stage-463123-e0
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD", "")
if not NEO4J_PASSWORD:
    raise ValueError("Set NEO4J_PASSWORD env var or retrieve from Secret Manager")

driver = GraphDatabase.driver(
    f"bolt://{NEO4J_IP}:7687",
    auth=("neo4j", NEO4J_PASSWORD),
)
driver.verify_connectivity()
print("Connected to Neo4j")


In [ ]:
# Verify seed data is loaded
with driver.session(database="neo4j") as session:
    result = session.run(
        "MATCH (n) RETURN labels(n)[0] AS label, count(n) AS cnt ORDER BY cnt DESC"
    )
    print("Node counts by label:")
    for row in result:
        print(f"  {row['label']}: {row['cnt']}")


In [ ]:
with driver.session(database="neo4j") as session:
    # Basic connectivity check
    result = session.run("MATCH (n) RETURN count(n) AS total")
    print(f"Total nodes: {result.single()['total']}")

    # Degree centrality via GDS
    result = session.run("""
        CALL gds.degree.stream('social-graph')
        YIELD nodeId, score
        RETURN gds.util.asNode(nodeId).id AS user_id, score AS degree
        ORDER BY degree DESC
        LIMIT 10
    """)
    for row in result:
        print(f"  {row['user_id']}: degree={row['degree']}")

In [ ]:
# Crear proyeccion del grafo social
with driver.session(database="neo4j") as session:
    # Proyeccion en memoria
    session.run("""
        CALL gds.graph.project(
            'social-graph',
            'Usuario',
            {
                CONOCE_A: {
                    orientation: 'UNDIRECTED',
                    properties: ['tie_strength']
                }
            }
        )
    """)

    # Louvain community detection
    result = session.run("""
        CALL gds.louvain.stream('social-graph')
        YIELD nodeId, communityId
        RETURN communityId, count(*) AS members
        ORDER BY members DESC
        LIMIT 10
    """)

    print("Top 10 comunidades (Louvain):")
    for row in result:
        print(f"  comunidad {row['communityId']}: {row['members']} miembros")

In [ ]:
# Degree centrality â€” usuarios mas conectados
with driver.session(database="neo4j") as session:
    result = session.run("""
        CALL gds.degree.stream('social-graph')
        YIELD nodeId, score
        MATCH (u:Usuario) WHERE id(u) = nodeId
        RETURN u.id AS user_id, score AS degree
        ORDER BY degree DESC
        LIMIT 10
    """)

    print("Top 10 usuarios por grado (CONOCE_A):")
    for row in result:
        print(f"  {row['user_id']}: {int(row['degree'])} conexiones")

## Firewall whitelist — `181.78.0.0/24`

The GCP firewall rule `allow-neo4j-bolt` permits Bolt (port 7687) only from `181.78.0.0/24`.  
If your public IP is outside that range you have two options:

**Option A — Add your IP to the whitelist** (requires `terraform apply`):
1. Edit `infra/terraform.tfvars` and add your IP/CIDR to `allowed_neo4j_ips`.
2. Run `terraform apply` from the `infra/` directory.

**Option B — Cloud IAP tunnel** (no firewall change needed, recommended):
```bash
gcloud compute start-iap-tunnel neo4j-vm 7687 \n  --local-host-port=localhost:7687 \n  --project=engaged-stage-463123-e0 \n  --zone=us-central1-a
```
Then set `NEO4J_IP = "localhost"` in the connection cell.

**Option C — Colab** (rotating IPs):
Colab IPs change between sessions. Use IAP tunnel via a local proxy, or add the Colab IP range to the whitelist each session.
